# 05 — Trajectory Inference (PAGA + Diffusion Pseudotime)

Computes trajectory topology and pseudotime ordering on the integrated annotated object using PAGA (partition-based graph abstraction) and diffusion pseudotime (DPT).

**Approach:**
- **PAGA** reveals which cell types are connected in the developmental trajectory — the topology of differentiation
- **Diffusion pseudotime (DPT)** assigns each cell an ordering value measuring how far it has progressed from the root state (vRG progenitors) along the trajectory
- Both methods are native to scanpy and work directly on the Harmony-corrected neighbor graph

**Key questions:**
1. What is the trajectory topology? Which cell types connect to which?
2. Do organoid cells reach the same pseudotime (maturation) as fetal cells?
3. Does pseudotime correlate with real gestational age (Zhong fetal data)?
4. Are the same genes activated at the same pseudotime in organoids vs fetal brain?

| Item | Value |
|------|-------|
| Input | `integrated_annotated.h5ad` (226,659 cells × 2,000 HVGs; raw: 14,498 genes) |
| Methods | PAGA + diffusion pseudotime (scanpy) |
| Root | vRG (ventricular radial glia) — most primitive neural progenitor |
| Datasets | Bhaduri 2020 (organoids) + Zhong 2018 (fetal PFC) |

**Prerequisites:** `colab_04_cell_type_annotation.ipynb` must have been run. `integrated_annotated.h5ad` must be on Drive.

**Runtime:** **High-RAM** recommended — diffusion map eigendecomposition on 226k cells is memory-intensive. Total runtime ~30–60 min.

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install scanpy and leidenalg
!pip install -q scanpy leidenalg

In [ ]:
# Import libraries and define all input/output paths
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import issparse

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white')

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'

PATHS = {
    'annotated':  os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_annotated.h5ad'),
    'trajectory': os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_trajectory.h5ad'),
}

print('Paths configured.')
for k, v in PATHS.items():
    print(f'  {k}: {v}')

## 2. Load Annotated Object

In [ ]:
# Load the annotated integrated object from colab_04
adata = sc.read_h5ad(PATHS['annotated'])

# Remove unused categories (safety for PAGA)
adata.obs['cell_type_integrated'] = adata.obs['cell_type_integrated'].cat.remove_unused_categories()

print(f'Object: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes (HVGs)')
print(f'Raw:    {adata.raw.shape[0]:,} cells x {adata.raw.shape[1]:,} genes (all shared)')
print()
print('obs columns:', list(adata.obs.columns))
print('obsm keys:  ', list(adata.obsm.keys()))
print('obsp keys:  ', list(adata.obsp.keys()))
print()
print('Cell types:', adata.obs['cell_type_integrated'].nunique())
print(adata.obs['cell_type_integrated'].value_counts())
print()
print('Datasets:', adata.obs['dataset'].value_counts().to_dict())

In [ ]:
# Quick orientation — current integrated UMAP
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc.pl.umap(adata, color='cell_type_integrated', ax=axes[0], show=False,
           title='Cell type', legend_loc='none')
sc.pl.umap(adata, color='dataset', ax=axes[1], show=False,
           title='Dataset')

plt.tight_layout()
plt.show()

## 3. Verify Neighbor Graph

The neighbor graph (built on `X_pca_harmony` in colab_03) is required for PAGA and diffusion pseudotime. It should be stored in the h5ad, but we verify and rebuild if needed.

In [ ]:
# Check that the Harmony-corrected neighbor graph is present
has_conn = 'connectivities' in adata.obsp
has_dist = 'distances' in adata.obsp

print(f'Connectivities in obsp: {has_conn}')
print(f'Distances in obsp:      {has_dist}')

if not has_conn or not has_dist:
    print()
    print('Neighbor graph not found — rebuilding from X_pca_harmony...')
    sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=30)
    print('Neighbor graph rebuilt.')
else:
    print()
    n_neighbors = int(adata.obsp['connectivities'][0].sum())
    print(f'Neighbor graph present (approx k={n_neighbors} per cell).')
    print('Ready for PAGA and DPT.')

## 4. PAGA — Trajectory Topology

PAGA (Partition-based graph abstraction) quantifies connectivity between cell type groups in the k-NN graph. Strong edges indicate a developmental relationship: cells of type A frequently neighbor cells of type B.

This gives us the **topology** of differentiation — which cell types connect, which are isolated, and where the trajectory branches.

In [ ]:
# Compute PAGA on cell type annotations
sc.tl.paga(adata, groups='cell_type_integrated')
print('PAGA computed.')
print(f'Connectivity matrix shape: {adata.uns["paga"]["connectivities"].shape}')

In [ ]:
# PAGA abstract graph — numbered labels with side legend
sc.pl.paga(adata, threshold=0.05, show=False)  # compute positions only
plt.close()  # suppress the default plot

cell_types = adata.obs['cell_type_integrated'].cat.categories.tolist()
paga_conn = adata.uns['paga']['connectivities'].toarray()
pos = adata.uns['paga']['pos']
palette = sc.pl.palettes.default_20 + sc.pl.palettes.default_20
threshold = 0.05

fig, ax = plt.subplots(figsize=(10, 8))

# Draw edges (thickness = connectivity strength)
for i in range(len(cell_types)):
    for j in range(i + 1, len(cell_types)):
        w = paga_conn[i, j]
        if w > threshold:
            lw = np.clip(w * 8, 0.5, 6)
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                    color='black', alpha=0.4, linewidth=lw, zorder=1)

# Draw nodes with numbers (size proportional to cell count)
node_sizes = adata.obs['cell_type_integrated'].value_counts()[cell_types].values
node_sizes_scaled = 100 + (node_sizes / node_sizes.max()) * 600

for k, ct in enumerate(cell_types):
    ax.scatter(pos[k, 0], pos[k, 1], s=node_sizes_scaled[k],
               color=palette[k % len(palette)], edgecolors='black',
               linewidth=0.5, zorder=2)
    ax.text(pos[k, 0], pos[k, 1], str(k + 1), fontsize=8, fontweight='bold',
            ha='center', va='center', zorder=3)

# Side legend
legend_text = '\n'.join([f'{k+1:2d}. {ct} ({node_sizes[k]:,})' for k, ct in enumerate(cell_types)])
ax.text(1.02, 0.98, legend_text, transform=ax.transAxes,
        fontsize=7, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('PAGA — cell type connectivity (threshold=0.05)')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# PAGA edges overlaid on the UMAP embedding
# Shows trajectory connections in the spatial context of the UMAP
fig, ax = plt.subplots(figsize=(14, 10))

umap = adata.obsm['X_umap']
ax.scatter(umap[:, 0], umap[:, 1], s=0.3, alpha=0.08, color='lightgrey', rasterized=True)

# Compute cell type centroids on UMAP
cell_types = adata.obs['cell_type_integrated'].cat.categories.tolist()
centroids = {}
for ct in cell_types:
    mask = (adata.obs['cell_type_integrated'] == ct).values
    centroids[ct] = umap[mask].mean(axis=0)

# Draw PAGA edges (strength > threshold)
paga_conn = adata.uns['paga']['connectivities'].toarray()
threshold = 0.05
for i in range(len(cell_types)):
    for j in range(i + 1, len(cell_types)):
        w = paga_conn[i, j]
        if w > threshold:
            ci, cj = centroids[cell_types[i]], centroids[cell_types[j]]
            lw = np.clip(w * 5, 0.5, 4)
            ax.plot([ci[0], cj[0]], [ci[1], cj[1]],
                    color='black', alpha=0.6, linewidth=lw, zorder=2)

# Plot centroids with numbered labels
palette = sc.pl.palettes.default_20 + sc.pl.palettes.default_20
for k, ct in enumerate(cell_types):
    c = centroids[ct]
    ax.scatter(c[0], c[1], s=180, color=palette[k % len(palette)],
               edgecolors='black', linewidth=0.5, zorder=3)
    ax.text(c[0], c[1] + 0.3, str(k + 1), fontsize=10, fontweight='bold',
            ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.7, ec='none'))

# Side legend
legend_text = '\n'.join([f'{k+1}. {ct}' for k, ct in enumerate(cell_types)])
ax.text(1.02, 0.98, legend_text, transform=ax.transAxes,
        fontsize=8, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('PAGA trajectory edges on UMAP')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. PAGA-Initialized UMAP

Recompute UMAP using PAGA node positions as initialization. This gives a layout that better reflects the trajectory topology — connected cell types are pulled closer together, and the overall shape is more tree-like.

The original Harmony UMAP is stored as `X_umap_harmony` for later comparison.

**Note:** UMAP recomputation on 226k cells takes ~5–15 minutes.

In [ ]:
# Store the current Harmony UMAP before recomputing
adata.obsm['X_umap_harmony'] = adata.obsm['X_umap'].copy()
print('Original UMAP stored as X_umap_harmony.')

# Recompute UMAP with PAGA initialization
sc.tl.umap(adata, init_pos='paga')
print('PAGA-initialized UMAP computed (stored as X_umap).')

In [ ]:
# Compare: Harmony UMAP vs PAGA-initialized UMAP
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sc.pl.embedding(adata, basis='umap_harmony', color='cell_type_integrated',
                ax=axes[0], show=False, title='Harmony UMAP (original)',
                legend_loc='none')

sc.pl.umap(adata, color='cell_type_integrated', ax=axes[1], show=False,
           title='PAGA-initialized UMAP', legend_loc='right margin')

plt.tight_layout()
plt.show()

## 6. Diffusion Map

Diffusion maps embed cells in a space where distances reflect random walk distances on the k-NN graph. The leading diffusion components capture the major axes of cellular variation — typically the main differentiation trajectories.

Diffusion pseudotime (next section) is computed from these components.

**Note:** Eigendecomposition on 226k cells may take 5–20 minutes on High-RAM.

In [ ]:
# Compute diffusion map on the Harmony-corrected neighbor graph
sc.tl.diffmap(adata, n_comps=15)
print('Diffusion map computed.')
print(f'Diffusion components shape: {adata.obsm["X_diffmap"].shape}')
print(f'Eigenvalues (top 5): {adata.uns["diffmap_evals"][:5]}')

In [ ]:
# Add leading DCs to obs for UMAP visualization
for i in range(4):
    adata.obs[f'DC{i+1}'] = adata.obsm['X_diffmap'][:, i]

# DCs on the PAGA-initialized UMAP
# DC1 should capture the main progenitor→neuron axis
sc.pl.umap(adata, color=['DC1', 'DC2', 'DC3', 'DC4'], ncols=2,
           title=['DC1 (main axis)', 'DC2', 'DC3', 'DC4'])

In [ ]:
# Diffusion map embedding — DC1 vs DC2, colored by cell type and dataset
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.embedding(adata, basis='diffmap', color='cell_type_integrated',
                components='1,2', ax=axes[0], show=False,
                title='Diffusion map — cell type', legend_loc='none')

sc.pl.embedding(adata, basis='diffmap', color='dataset',
                components='1,2', ax=axes[1], show=False,
                title='Diffusion map — dataset')

plt.tight_layout()
plt.show()

## 7. Select Root Cell

Diffusion pseudotime measures the distance of each cell from a designated root cell along the diffusion map. The root should be the most primitive progenitor state.

**Strategy:** Select a vRG (ventricular radial glia) cell at the extreme progenitor end of diffusion component 1. vRG cells are SOX2+/PAX6+ — the earliest neural stem cells in cortical development.

In [ ]:
# Find vRG cells
vrg_mask = (adata.obs['cell_type_integrated'] == 'vRG').values
vrg_indices = np.where(vrg_mask)[0]
print(f'vRG cells: {len(vrg_indices):,}')

# DC1 should separate progenitors from neurons — check which direction is "early"
dc1 = adata.obsm['X_diffmap'][:, 0]

# Compare DC1 values in vRG vs mature neurons
neuron_types = ['Excitatory neurons', 'Excitatory neurons (mature)', 'Mature neurons']
neuron_mask = adata.obs['cell_type_integrated'].isin(neuron_types).values

vrg_dc1_med = np.median(dc1[vrg_mask])
neuron_dc1_med = np.median(dc1[neuron_mask])

print(f'DC1 median — vRG: {vrg_dc1_med:.4f}')
print(f'DC1 median — neurons: {neuron_dc1_med:.4f}')

# Root = vRG cell at the extreme progenitor end of DC1
if vrg_dc1_med > neuron_dc1_med:
    root_idx = vrg_indices[np.argmax(dc1[vrg_indices])]
    print('vRG has higher DC1 → selecting vRG with max DC1 as root')
else:
    root_idx = vrg_indices[np.argmin(dc1[vrg_indices])]
    print('vRG has lower DC1 → selecting vRG with min DC1 as root')

adata.uns['iroot'] = root_idx

print(f'\nRoot cell index: {root_idx}')
print(f'Root cell barcode: {adata.obs_names[root_idx]}')
print(f'Root cell dataset: {adata.obs["dataset"].iloc[root_idx]}')
print(f'Root cell DC1: {dc1[root_idx]:.4f}')

## 8. Diffusion Pseudotime

Computes pseudotime as the geodesic distance from the root cell on the diffusion map. Low pseudotime = progenitor-like, high pseudotime = mature.

Cells in disconnected graph components (if any) get infinite pseudotime — these are replaced with NaN for plotting.

In [ ]:
# Compute diffusion pseudotime from the root cell
sc.tl.dpt(adata, n_dcs=15)

# Handle disconnected components
n_inf = np.isinf(adata.obs['dpt_pseudotime']).sum()
if n_inf > 0:
    print(f'WARNING: {n_inf:,} cells with infinite pseudotime (disconnected from root)')
    # Identify which cell types are affected
    inf_mask = np.isinf(adata.obs['dpt_pseudotime'])
    print('Affected cell types:')
    print(adata.obs.loc[inf_mask, 'cell_type_integrated'].value_counts().to_string())
    # Replace inf with NaN for cleaner plotting
    adata.obs['dpt_pseudotime'] = adata.obs['dpt_pseudotime'].replace(np.inf, np.nan)
    valid = adata.obs['dpt_pseudotime'].notna().sum()
    print(f'Replaced inf → NaN. Valid pseudotime: {valid:,} / {adata.shape[0]:,} cells')
else:
    print('All cells have finite pseudotime (graph is fully connected).')

print(f'\nPseudotime range: {adata.obs["dpt_pseudotime"].min():.4f} — '
      f'{adata.obs["dpt_pseudotime"].max():.4f}')

In [ ]:
# Pseudotime on UMAP — both the PAGA-initialized and original Harmony layouts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(adata, color='dpt_pseudotime', ax=axes[0], show=False,
           title='Pseudotime (PAGA UMAP)', color_map='viridis')

sc.pl.embedding(adata, basis='umap_harmony', color='dpt_pseudotime',
                ax=axes[1], show=False,
                title='Pseudotime (Harmony UMAP)', color_map='viridis')

plt.tight_layout()
plt.show()

In [ ]:
# Pseudotime on UMAP — split by dataset
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, dataset, title in zip(axes,
                               ['bhaduri', 'zhong'],
                               ['Organoid (Bhaduri)', 'Fetal (Zhong)']):
    mask = (adata.obs['dataset'] == dataset).values
    # Background: other dataset in grey
    ax.scatter(adata.obsm['X_umap'][~mask, 0], adata.obsm['X_umap'][~mask, 1],
               s=0.1, alpha=0.03, color='lightgrey', rasterized=True)
    # This dataset colored by pseudotime
    valid = mask & adata.obs['dpt_pseudotime'].notna().values
    sc_plot = ax.scatter(
        adata.obsm['X_umap'][valid, 0], adata.obsm['X_umap'][valid, 1],
        c=adata.obs.loc[valid, 'dpt_pseudotime'].values,
        s=0.3, alpha=0.5, cmap='viridis', rasterized=True)
    plt.colorbar(sc_plot, ax=ax, shrink=0.6, label='Pseudotime')
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 9. Pseudotime by Cell Type

Violin plots showing pseudotime distribution per cell type, ordered by median pseudotime. The ordering should recapitulate known cortical development: vRG → cycling progenitors → oRG → immature neurons → mature neurons.

In [ ]:
# Order cell types by median pseudotime
ct_medians = (adata.obs.groupby('cell_type_integrated')['dpt_pseudotime']
              .median().sort_values())
ct_order = ct_medians.index.tolist()

fig, ax = plt.subplots(figsize=(16, 6))
sc.pl.violin(adata, keys='dpt_pseudotime', groupby='cell_type_integrated',
             order=ct_order, rotation=90, ax=ax, show=False, stripplot=False)
ax.set_title('Pseudotime by cell type (ordered by median)')
ax.set_ylabel('Diffusion pseudotime')
plt.tight_layout()
plt.show()

In [ ]:
# Print median pseudotime per cell type — full table
print('Median pseudotime per cell type (ascending):')
print('=' * 65)
for ct in ct_order:
    mask = adata.obs['cell_type_integrated'] == ct
    median_pt = adata.obs.loc[mask, 'dpt_pseudotime'].median()
    n_total = mask.sum()
    n_valid = adata.obs.loc[mask, 'dpt_pseudotime'].notna().sum()
    print(f'  {ct:42s}  median={median_pt:.4f}  n={n_total:>7,}  (valid={n_valid:,})')

## 10. Organoid vs Fetal Pseudotime Comparison

The central trajectory question: do organoid cells reach the same maturation level (pseudotime) as fetal cells?

If organoids are "stuck" in progenitor states (as suggested by the composition analysis in colab_04), their pseudotime distribution should be shifted toward lower values compared to fetal cells.

In [ ]:
# Overall pseudotime distribution: organoid vs fetal
fig, ax = plt.subplots(figsize=(10, 5))
for dataset, color, label in [('bhaduri', '#1f77b4', 'Organoid (Bhaduri)'),
                               ('zhong', '#ff7f0e', 'Fetal (Zhong)')]:
    mask = (adata.obs['dataset'] == dataset) & adata.obs['dpt_pseudotime'].notna()
    ax.hist(adata.obs.loc[mask, 'dpt_pseudotime'], bins=100, alpha=0.5,
            color=color, label=f'{label} (n={mask.sum():,})', density=True)
ax.set_xlabel('Diffusion pseudotime')
ax.set_ylabel('Density')
ax.set_title('Pseudotime distribution — organoid vs fetal')
ax.legend()
plt.tight_layout()
plt.show()

# Mann-Whitney U test
bhaduri_pt = adata.obs.loc[adata.obs['dataset'] == 'bhaduri', 'dpt_pseudotime'].dropna()
zhong_pt = adata.obs.loc[adata.obs['dataset'] == 'zhong', 'dpt_pseudotime'].dropna()
U, p = stats.mannwhitneyu(bhaduri_pt, zhong_pt, alternative='two-sided')
print(f'Mann-Whitney U test: U={U:.0f}, p={p:.2e}')
print(f'Organoid median pseudotime: {bhaduri_pt.median():.4f}')
print(f'Fetal median pseudotime:    {zhong_pt.median():.4f}')

In [ ]:
# Per cell type: median pseudotime comparison (organoid vs fetal)
# Only include cell types with >= 10 cells from each dataset
MIN_CELLS = 10
comparison_rows = []
for ct in adata.obs['cell_type_integrated'].cat.categories:
    for dataset in ['bhaduri', 'zhong']:
        mask = ((adata.obs['cell_type_integrated'] == ct) &
                (adata.obs['dataset'] == dataset) &
                adata.obs['dpt_pseudotime'].notna())
        n = mask.sum()
        if n > 0:
            comparison_rows.append({
                'cell_type': ct,
                'dataset': dataset,
                'median_pt': adata.obs.loc[mask, 'dpt_pseudotime'].median(),
                'mean_pt': adata.obs.loc[mask, 'dpt_pseudotime'].mean(),
                'n_cells': n,
            })

comp_df = pd.DataFrame(comparison_rows)

# Print full table
print('Median pseudotime — organoid vs fetal per cell type:')
print('=' * 90)
for ct in ct_order:
    row_b = comp_df[(comp_df['cell_type'] == ct) & (comp_df['dataset'] == 'bhaduri')]
    row_z = comp_df[(comp_df['cell_type'] == ct) & (comp_df['dataset'] == 'zhong')]
    b_str = f'{row_b.iloc[0]["median_pt"]:.4f} (n={int(row_b.iloc[0]["n_cells"]):,})' if len(row_b) > 0 else '—'
    z_str = f'{row_z.iloc[0]["median_pt"]:.4f} (n={int(row_z.iloc[0]["n_cells"]):,})' if len(row_z) > 0 else '—'
    print(f'  {ct:42s}  Org: {b_str:22s}  Fetal: {z_str}')

In [ ]:
# Scatter plot: median pseudotime per cell type — organoid (x) vs fetal (y)
# Cell types near the diagonal have similar maturation in both conditions
both = comp_df.groupby('cell_type').filter(lambda x: len(x) == 2 and (x['n_cells'] >= MIN_CELLS).all())

if len(both) > 0:
    pivot = both.pivot(index='cell_type', columns='dataset', values='median_pt')

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(pivot['bhaduri'], pivot['zhong'], s=80, c='steelblue',
               edgecolors='black', linewidth=0.5, zorder=3)

    for ct in pivot.index:
        ax.annotate(ct, (pivot.loc[ct, 'bhaduri'], pivot.loc[ct, 'zhong']),
                    fontsize=7, xytext=(5, 5), textcoords='offset points')

    # Diagonal (equal pseudotime)
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, label='Equal pseudotime')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_xlabel('Organoid median pseudotime')
    ax.set_ylabel('Fetal median pseudotime')
    ax.set_title('Cell type maturation: organoid vs fetal')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'{len(pivot)} cell types with >= {MIN_CELLS} cells from each dataset')
else:
    print(f'No cell types with >= {MIN_CELLS} cells from both datasets.')
    print('Reducing threshold to MIN_CELLS=1 for overview:')
    both_any = comp_df.groupby('cell_type').filter(lambda x: len(x) == 2)
    if len(both_any) > 0:
        pivot_any = both_any.pivot(index='cell_type', columns='dataset', values='median_pt')
        print(pivot_any.round(4).to_string())

## 11. Pseudotime vs Gestational Week (Validation)

For the Zhong (fetal) cells, pseudotime should correlate with real gestational age. This is the strongest validation that diffusion pseudotime captures genuine developmental progression rather than technical artifacts.

**Expected:** positive Spearman correlation — early GW (GW8–10) = low pseudotime, late GW (GW23–26) = high pseudotime.

In [ ]:
# Zhong cells: pseudotime vs gestational week
zhong_mask = (adata.obs['dataset'] == 'zhong') & adata.obs['dpt_pseudotime'].notna()
zhong_obs = adata.obs[zhong_mask].copy()

if 'gestational_week' in zhong_obs.columns and zhong_obs['gestational_week'].notna().sum() > 0:
    # Extract numeric GW for correlation
    zhong_obs['gw_numeric'] = zhong_obs['gestational_week'].str.extract(r'(\d+)').astype(float)

    # Box plot: pseudotime by gestational week
    gw_order = sorted(zhong_obs['gestational_week'].dropna().unique())

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Box plot
    ax = axes[0]
    data_by_gw = [zhong_obs.loc[zhong_obs['gestational_week'] == gw, 'dpt_pseudotime'].dropna().values
                  for gw in gw_order]
    bp = ax.boxplot(data_by_gw, labels=gw_order, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('#ff7f0e')
        patch.set_alpha(0.5)
    ax.set_xlabel('Gestational week')
    ax.set_ylabel('Diffusion pseudotime')
    ax.set_title('Pseudotime vs gestational week (Zhong fetal)')

    # Scatter with regression trend
    ax = axes[1]
    valid = zhong_obs[['gw_numeric', 'dpt_pseudotime']].dropna()
    ax.scatter(valid['gw_numeric'], valid['dpt_pseudotime'], s=5, alpha=0.3, color='#ff7f0e')
    # Add per-GW medians as large points
    gw_medians = valid.groupby('gw_numeric')['dpt_pseudotime'].median()
    ax.scatter(gw_medians.index, gw_medians.values, s=100, color='red',
               edgecolors='black', zorder=5, label='Median per GW')
    ax.set_xlabel('Gestational week (numeric)')
    ax.set_ylabel('Diffusion pseudotime')
    ax.set_title('Pseudotime vs GW — per-cell scatter')
    ax.legend()

    plt.tight_layout()
    plt.show()

    # Spearman correlation
    rho, p = stats.spearmanr(valid['gw_numeric'], valid['dpt_pseudotime'])
    print(f'Spearman correlation (GW vs pseudotime): rho={rho:.3f}, p={p:.2e}')
    print()
    print('Median pseudotime per gestational week:')
    for gw in gw_order:
        gw_sub = zhong_obs[zhong_obs['gestational_week'] == gw]
        n = len(gw_sub)
        med = gw_sub['dpt_pseudotime'].median()
        print(f'  {gw}: median={med:.4f}  n={n}')
else:
    print('gestational_week not available for Zhong cells.')

## 12. Gene Expression Dynamics Along Pseudotime

Key marker genes plotted against pseudotime, split by dataset. If organoids and fetal brain follow the same transcriptional trajectory, the same genes should activate (or silence) at similar pseudotime positions.

**Expected patterns:**
- Progenitor genes (SOX2, PAX6, VIM) — high early, decreasing with pseudotime
- Neuronal genes (NEUROD2, TBR1, BCL11B, STMN2) — low early, increasing
- Interneuron genes (DLX2, GAD1) — branch-specific activation
- oRG / glial genes (HOPX, GFAP) — specific to the glial branch

In [ ]:
# Define trajectory genes to plot
TRAJECTORY_GENES = [
    # Progenitor markers (should decrease)
    'SOX2', 'PAX6', 'VIM', 'HES1',
    # Neuronal maturation (should increase)
    'NEUROD2', 'TBR1', 'BCL11B', 'STMN2',
    # Interneuron (branch-specific)
    'DLX2', 'GAD1',
    # oRG / glial
    'HOPX', 'GFAP',
    # Cycling
    'MKI67',
]

# Filter to available genes
available_genes = [g for g in TRAJECTORY_GENES if g in adata.raw.var_names]
missing_genes = [g for g in TRAJECTORY_GENES if g not in adata.raw.var_names]

print(f'Available: {len(available_genes)} / {len(TRAJECTORY_GENES)}')
if missing_genes:
    print(f'Missing: {missing_genes}')

In [ ]:
# Build gene name → index lookup for efficient access to adata.raw
gene_to_idx = {g: i for i, g in enumerate(adata.raw.var_names)}

# Get pseudotime and dataset for cells with valid pseudotime
valid_mask = adata.obs['dpt_pseudotime'].notna().values
pt_values = adata.obs.loc[valid_mask, 'dpt_pseudotime'].values
ds_values = adata.obs.loc[valid_mask, 'dataset'].values

# Extract raw expression for each gene
raw_X = adata.raw.X[valid_mask]
raw_expr = {}
for gene in available_genes:
    col = raw_X[:, gene_to_idx[gene]]
    if issparse(col):
        col = col.toarray().ravel()
    else:
        col = np.asarray(col).ravel()
    raw_expr[gene] = col

# Plot: binned mean expression along pseudotime, split by dataset
n_bins = 80
pt_bins = np.linspace(np.nanmin(pt_values), np.nanmax(pt_values), n_bins + 1)
bin_idx = np.digitize(pt_values, pt_bins) - 1
bin_idx = np.clip(bin_idx, 0, n_bins - 1)
bin_centers = (pt_bins[:-1] + pt_bins[1:]) / 2

n_genes = len(available_genes)
fig, axes = plt.subplots(n_genes, 1, figsize=(10, 2.5 * n_genes), sharex=True)
if n_genes == 1:
    axes = [axes]

for ax, gene in zip(axes, available_genes):
    expr = raw_expr[gene]
    for dataset, color in [('bhaduri', '#1f77b4'), ('zhong', '#ff7f0e')]:
        ds_mask = ds_values == dataset
        bin_means = np.full(n_bins, np.nan)
        for b in range(n_bins):
            b_mask = (bin_idx == b) & ds_mask
            if b_mask.sum() > 5:  # require at least 5 cells per bin
                bin_means[b] = expr[b_mask].mean()
        ax.plot(bin_centers, bin_means, color=color, label=dataset.capitalize(),
                alpha=0.8, linewidth=1.5)
    ax.set_ylabel(gene, fontsize=10, fontweight='bold')
    if gene == available_genes[0]:
        ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Diffusion pseudotime')
plt.suptitle('Gene expression dynamics along pseudotime', fontsize=13, y=1.005)
plt.tight_layout()
plt.show()

## 13. Lineage-Specific Analysis

Focus on the two main neuronal lineages separately:

**Excitatory neuron lineage:** vRG → cortical progenitors → immature neurons → excitatory / mature neurons
**GABAergic interneuron lineage:** vRG → cycling progenitors → GABAergic interneurons

Examining pseudotime within each lineage gives a cleaner view of maturation than the full-object pseudotime, which mixes all branches.

In [ ]:
# Define lineage cell types (matching colab_04 annotations exactly)
EXCITATORY_LINEAGE = [
    'vRG', 'Cortical progenitors (EMX2+)', 'Cycling cortical progenitors',
    'Immature neurons (early)', 'Immature neurons (migrating)',
    'Excitatory neurons', 'Excitatory neurons (mature)',
    'Excitatory neurons (maturing)', 'Mature neurons',
]

GABAERGIC_LINEAGE = [
    'vRG', 'Cycling progenitors (G2/M)', 'Cycling progenitors (S-phase)',
    'Cycling progenitors', 'GABAergic interneurons',
]

# Filter to cell types that exist in this dataset
existing = set(adata.obs['cell_type_integrated'].cat.categories)
exc_types = [ct for ct in EXCITATORY_LINEAGE if ct in existing]
gaba_types = [ct for ct in GABAERGIC_LINEAGE if ct in existing]

print(f'Excitatory lineage: {len(exc_types)} cell types')
for ct in exc_types:
    n = (adata.obs['cell_type_integrated'] == ct).sum()
    print(f'  {ct}: {n:,}')

print(f'\nGABAergic lineage: {len(gaba_types)} cell types')
for ct in gaba_types:
    n = (adata.obs['cell_type_integrated'] == ct).sum()
    print(f'  {ct}: {n:,}')

In [ ]:
# Excitatory lineage — pseudotime analysis
exc_mask = (adata.obs['cell_type_integrated'].isin(exc_types) &
            adata.obs['dpt_pseudotime'].notna())
exc_obs = adata.obs[exc_mask].copy()

exc_order = (exc_obs.groupby('cell_type_integrated')['dpt_pseudotime']
             .median().sort_values().index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Violin by cell type (ordered by pseudotime)
ax = axes[0]
positions = range(len(exc_order))
for i, ct in enumerate(exc_order):
    vals = exc_obs.loc[exc_obs['cell_type_integrated'] == ct, 'dpt_pseudotime'].values
    parts = ax.violinplot([vals], positions=[i], showmedians=True, widths=0.7)
    for pc in parts['bodies']:
        pc.set_alpha(0.5)
ax.set_xticks(list(positions))
ax.set_xticklabels(exc_order, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Pseudotime')
ax.set_title('Excitatory lineage — pseudotime by cell type')

# Organoid vs fetal within excitatory lineage
ax = axes[1]
for dataset, color in [('bhaduri', '#1f77b4'), ('zhong', '#ff7f0e')]:
    mask = exc_obs['dataset'] == dataset
    if mask.sum() > 0:
        ax.hist(exc_obs.loc[mask, 'dpt_pseudotime'], bins=80, alpha=0.5,
                color=color, label=f'{dataset.capitalize()} (n={mask.sum():,})', density=True)
ax.set_xlabel('Pseudotime')
ax.set_ylabel('Density')
ax.set_title('Excitatory lineage — organoid vs fetal')
ax.legend()

plt.tight_layout()
plt.show()

print('Excitatory lineage median pseudotime:')
for ct in exc_order:
    med = exc_obs.loc[exc_obs['cell_type_integrated'] == ct, 'dpt_pseudotime'].median()
    print(f'  {ct:42s}  {med:.4f}')

In [ ]:
# GABAergic lineage — pseudotime analysis
gaba_mask = (adata.obs['cell_type_integrated'].isin(gaba_types) &
             adata.obs['dpt_pseudotime'].notna())
gaba_obs = adata.obs[gaba_mask].copy()

gaba_order = (gaba_obs.groupby('cell_type_integrated')['dpt_pseudotime']
              .median().sort_values().index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for i, ct in enumerate(gaba_order):
    vals = gaba_obs.loc[gaba_obs['cell_type_integrated'] == ct, 'dpt_pseudotime'].values
    parts = ax.violinplot([vals], positions=[i], showmedians=True, widths=0.7)
    for pc in parts['bodies']:
        pc.set_alpha(0.5)
ax.set_xticks(range(len(gaba_order)))
ax.set_xticklabels(gaba_order, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Pseudotime')
ax.set_title('GABAergic lineage — pseudotime by cell type')

ax = axes[1]
for dataset, color in [('bhaduri', '#1f77b4'), ('zhong', '#ff7f0e')]:
    mask = gaba_obs['dataset'] == dataset
    if mask.sum() > 0:
        ax.hist(gaba_obs.loc[mask, 'dpt_pseudotime'], bins=80, alpha=0.5,
                color=color, label=f'{dataset.capitalize()} (n={mask.sum():,})', density=True)
ax.set_xlabel('Pseudotime')
ax.set_ylabel('Density')
ax.set_title('GABAergic lineage — organoid vs fetal')
ax.legend()

plt.tight_layout()
plt.show()

print('GABAergic lineage median pseudotime:')
for ct in gaba_order:
    med = gaba_obs.loc[gaba_obs['cell_type_integrated'] == ct, 'dpt_pseudotime'].median()
    print(f'  {ct:42s}  {med:.4f}')

## 14. Per-Dataset Pseudotime (Independent Trajectories)

Run diffusion pseudotime separately on each dataset. This checks whether the trajectory structure holds when computed independently — not in the shared integrated space.

**Zhong (fetal, ~2.3k cells):** fast — a few seconds.
**Bhaduri (organoid, ~224k cells):** slow — diffusion map may take 10–30 min on High-RAM.

In [ ]:
# --- Zhong (fetal) — independent DPT ---
print('=== Zhong (fetal) — independent DPT ===')
zhong_sub = adata[adata.obs['dataset'] == 'zhong'].copy()
print(f'Cells: {zhong_sub.shape[0]:,}')

# Rebuild neighbor graph on this subset (fewer neighbors for small dataset)
sc.pp.neighbors(zhong_sub, use_rep='X_pca_harmony', n_neighbors=15)
sc.tl.diffmap(zhong_sub, n_comps=10)

# Root: vRG at extreme of DC1
zhong_types = zhong_sub.obs['cell_type_integrated'].values
zhong_vrg = np.where(zhong_types == 'vRG')[0]

if len(zhong_vrg) == 0:
    # Fallback: any progenitor type
    prog_types = [ct for ct in np.unique(zhong_types)
                  if 'RG' in ct or 'rogenitor' in ct.lower()]
    zhong_vrg = np.where(np.isin(zhong_types, prog_types))[0]
    print(f'No vRG in Zhong — using: {prog_types}')

dc1_z = zhong_sub.obsm['X_diffmap'][:, 0]
neuron_idx = np.where(np.isin(zhong_types, ['Excitatory neurons', 'Excitatory neurons (mature)', 'Mature neurons']))[0]

vrg_med = np.median(dc1_z[zhong_vrg]) if len(zhong_vrg) > 0 else 0
neur_med = np.median(dc1_z[neuron_idx]) if len(neuron_idx) > 0 else 0

if vrg_med > neur_med:
    root_z = zhong_vrg[np.argmax(dc1_z[zhong_vrg])]
else:
    root_z = zhong_vrg[np.argmin(dc1_z[zhong_vrg])]

zhong_sub.uns['iroot'] = root_z
sc.tl.dpt(zhong_sub, n_dcs=10)
zhong_sub.obs['dpt_pseudotime'] = zhong_sub.obs['dpt_pseudotime'].replace(np.inf, np.nan)
sc.tl.umap(zhong_sub)

print(f'Pseudotime range: {zhong_sub.obs["dpt_pseudotime"].min():.4f} — '
      f'{zhong_sub.obs["dpt_pseudotime"].max():.4f}')

In [ ]:
# Zhong independent: UMAP + pseudotime + gestational week
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.umap(zhong_sub, color='cell_type_integrated', ax=axes[0], show=False,
           title='Zhong — cell type', legend_loc='none')
sc.pl.umap(zhong_sub, color='dpt_pseudotime', ax=axes[1], show=False,
           title='Zhong — independent pseudotime', color_map='viridis')
if 'gestational_week' in zhong_sub.obs.columns:
    sc.pl.umap(zhong_sub, color='gestational_week', ax=axes[2], show=False,
               title='Zhong — gestational week')
else:
    axes[2].set_title('gestational_week not available')
    axes[2].axis('off')

plt.tight_layout()
plt.show()

# Independent pseudotime vs gestational week correlation
if 'gestational_week' in zhong_sub.obs.columns:
    zhong_ind = zhong_sub.obs.copy()
    zhong_ind['gw_numeric'] = zhong_ind['gestational_week'].str.extract(r'(\d+)').astype(float)
    valid = zhong_ind[['gw_numeric', 'dpt_pseudotime']].dropna()
    if len(valid) > 10:
        rho, p = stats.spearmanr(valid['gw_numeric'], valid['dpt_pseudotime'])
        print(f'Zhong independent — Spearman (GW vs pseudotime): rho={rho:.3f}, p={p:.2e}')

In [ ]:
# --- Bhaduri (organoid) — independent DPT ---
# This is slow — ~224k cells. Diffusion map eigendecomposition is the bottleneck.
print('=== Bhaduri (organoid) — independent DPT ===')
bhaduri_sub = adata[adata.obs['dataset'] == 'bhaduri'].copy()
print(f'Cells: {bhaduri_sub.shape[0]:,}')

# Rebuild neighbor graph
sc.pp.neighbors(bhaduri_sub, use_rep='X_pca_harmony', n_neighbors=30)
print('Neighbor graph built.')

# Diffusion map — this is the slow step
print('Computing diffusion map (this may take 10-30 min)...')
sc.tl.diffmap(bhaduri_sub, n_comps=10)
print('Diffusion map computed.')

# Root: vRG at extreme of DC1
bhaduri_types = bhaduri_sub.obs['cell_type_integrated'].values
bhaduri_vrg = np.where(bhaduri_types == 'vRG')[0]
dc1_b = bhaduri_sub.obsm['X_diffmap'][:, 0]

neuron_idx_b = np.where(np.isin(bhaduri_types, ['Excitatory neurons', 'Excitatory neurons (mature)', 'Mature neurons']))[0]
vrg_med_b = np.median(dc1_b[bhaduri_vrg])
neur_med_b = np.median(dc1_b[neuron_idx_b]) if len(neuron_idx_b) > 0 else 0

if vrg_med_b > neur_med_b:
    root_b = bhaduri_vrg[np.argmax(dc1_b[bhaduri_vrg])]
else:
    root_b = bhaduri_vrg[np.argmin(dc1_b[bhaduri_vrg])]

bhaduri_sub.uns['iroot'] = root_b
sc.tl.dpt(bhaduri_sub, n_dcs=10)
bhaduri_sub.obs['dpt_pseudotime'] = bhaduri_sub.obs['dpt_pseudotime'].replace(np.inf, np.nan)

print(f'Pseudotime range: {bhaduri_sub.obs["dpt_pseudotime"].min():.4f} — '
      f'{bhaduri_sub.obs["dpt_pseudotime"].max():.4f}')

In [ ]:
# Bhaduri independent: UMAP + pseudotime
# Recompute UMAP for the organoid-only subset
print('Computing organoid-only UMAP...')
sc.tl.umap(bhaduri_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc.pl.umap(bhaduri_sub, color='cell_type_integrated', ax=axes[0], show=False,
           title='Bhaduri — cell type', legend_loc='none')
sc.pl.umap(bhaduri_sub, color='dpt_pseudotime', ax=axes[1], show=False,
           title='Bhaduri — independent pseudotime', color_map='viridis')

plt.tight_layout()
plt.show()

# Cell type ordering by independent pseudotime
ct_order_b = (bhaduri_sub.obs.groupby('cell_type_integrated')['dpt_pseudotime']
              .median().sort_values().index.tolist())
print('\nBhaduri independent — median pseudotime per cell type:')
for ct in ct_order_b:
    mask = bhaduri_sub.obs['cell_type_integrated'] == ct
    n = mask.sum()
    med = bhaduri_sub.obs.loc[mask, 'dpt_pseudotime'].median()
    print(f'  {ct:42s}  median={med:.4f}  n={n:>7,}')

In [ ]:
# Compare cell type ordering: integrated vs per-dataset pseudotime
print('Cell type ordering comparison (by median pseudotime):')
print('=' * 80)
print(f'{"Integrated":42s}  {"Bhaduri independent":42s}')
print('-' * 80)

# Get integrated ordering from ct_order (computed in section 9)
for i in range(max(len(ct_order), len(ct_order_b))):
    int_ct = ct_order[i] if i < len(ct_order) else ''
    bh_ct = ct_order_b[i] if i < len(ct_order_b) else ''
    match = ' ✓' if int_ct == bh_ct else ''
    print(f'  {int_ct:42s}  {bh_ct:42s}{match}')

## 15. Save

In [ ]:
# Clean up temporary DC columns used for plotting (keep data in obsm)
for col in ['DC1', 'DC2', 'DC3', 'DC4']:
    if col in adata.obs.columns:
        del adata.obs[col]

# Save the trajectory-enriched AnnData to Drive
# New content vs colab_04:
#   obs: dpt_pseudotime
#   obsm: X_diffmap, X_umap_harmony (original), X_umap (PAGA-initialized)
#   uns: paga, diffmap_evals, iroot

adata.write_h5ad(PATHS['trajectory'])

size_mb = os.path.getsize(PATHS['trajectory']) / 1e6
print(f'Saved: {PATHS["trajectory"]} ({size_mb:.1f} MB)')
print()
print('Object summary:')
print(f'  Cells:            {adata.shape[0]:,}')
print(f'  Genes (HVG set):  {adata.shape[1]:,}')
print(f'  Genes (raw):      {adata.raw.shape[1]:,}')
print(f'  obs keys:         {list(adata.obs.columns)}')
print(f'  obsm keys:        {list(adata.obsm.keys())}')
print(f'  uns keys:         {[k for k in adata.uns.keys()]}')

## Done

Trajectory analysis complete. The integrated object now contains:

**New in obs:**
- `dpt_pseudotime` — diffusion pseudotime from vRG root (0 = progenitor, higher = more mature)

**New in obsm:**
- `X_diffmap` — diffusion map embedding (15 components)
- `X_umap_harmony` — original Harmony UMAP (from colab_03)
- `X_umap` — PAGA-initialized UMAP (better trajectory layout)

**New in uns:**
- `paga` — PAGA connectivity matrix between cell types
- `diffmap_evals` — diffusion map eigenvalues
- `iroot` — root cell index (vRG)

Saved to Drive: `data/processed/integrated/integrated_trajectory.h5ad`

**Key results to inspect:**
- Section 4 (PAGA): does the trajectory topology match known cortical development?
- Section 9 (pseudotime ordering): vRG → cycling → oRG → immature → mature neurons?
- Section 10 (organoid vs fetal): are organoid cells at lower pseudotime (less mature)?
- Section 11 (GW validation): does DPT correlate with real gestational age?
- Section 12 (gene dynamics): same genes activated at same pseudotime in both datasets?

**Next steps:**
- `colab_06_rna_velocity.ipynb` — scVelo (RNA velocity: direction of transcriptional change, requires spliced/unspliced count matrices)